# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
# Task 1 — fossil fuel treemap
df_fossil = df.loc[df["Source_Type"] == "Fossil"].copy()

# Aggregate to the hierarchy's lowest level to avoid duplicate leaves.
fossil_by_level = (
    df_fossil.groupby(["Region", "Country", "Source"], as_index=False)["TWh"]
    .sum()
)

# Identify the most fossil-dependent country by fossil TWh.
country_fossil = fossil_by_level.groupby("Country", as_index=False)["TWh"].sum()
top_fossil_country = country_fossil.loc[country_fossil["TWh"].idxmax()]
top_fossil_region = (
    fossil_by_level.groupby("Region")["TWh"].sum().idxmax()
)

# CVD-safe palette: blue, orange and green.
fossil_palette = {
    "Coal": "#0072B2",
    "Oil": "#E69F00",
    "Natural Gas": "#009E73"
}

fig1 = px.treemap(
    fossil_by_level,
    path=["Region", "Country", "Source"],
    values="TWh",
    color="Source",
    color_discrete_map=fossil_palette,
    title=(
        f"Fossil fuel production: {top_fossil_country['Country']} leads "
        f"({top_fossil_country['TWh']:,.0f} TWh); "
        f"{top_fossil_region} leads by region"
    )
)

# Keep source leaves coloured and grey out Region/Country parent nodes.
trace = fig1.data[0]
trace.marker.colors = [
    fossil_palette.get(label, "#BDBDBD") for label in trace.labels
]
trace.texttemplate = "%{label}<br>%{value:,.0f} TWh"
trace.hovertemplate = (
    "<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>"
)
fig1.update_layout(margin=dict(t=80, l=10, r=10, b=10))
fig1.show()

print("Largest fossil-fuel country by TWh:")
print(top_fossil_country.to_string(index=False))
print(f"Region with the greatest fossil TWh total: {top_fossil_region}")

Largest fossil-fuel country by TWh:
Saudi Arabia
      9671.0
Region with the greatest fossil TWh total: Asia


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [3]:
# Task 2 — low-carbon treemap and horizontal bar chart
df_low_carbon = df.loc[df["Source_Type"] == "Low-carbon"].copy()

low_carbon_by_country = (
    df_low_carbon.groupby("Country", as_index=False)["TWh"]
    .sum()
    .sort_values("TWh", ascending=False)
)

# Add the required dummy root for a single-level hierarchy.
low_carbon_by_country["All"] = "Low-carbon"
leading_country = low_carbon_by_country.iloc[0]

fig2_tree = px.treemap(
    low_carbon_by_country,
    path=["All", "Country"],
    values="TWh",
    color="TWh",
    color_continuous_scale="Blues",
    title="Low-carbon energy (Nuclear + Hydro) by country"
)
fig2_tree.update_traces(
    texttemplate="%{label}<br>%{value:,.0f} TWh",
    hovertemplate="<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>"
)
fig2_tree.update_layout(margin=dict(t=60, l=10, r=10, b=10))
fig2_tree.show()

fig2_bar = px.bar(
    low_carbon_by_country.sort_values("TWh", ascending=True),
    x="TWh",
    y="Country",
    orientation="h",
    color_discrete_sequence=["#0072B2"],
    title=(
        f"Low-carbon energy by country — leader: "
        f"{leading_country['Country']} ({leading_country['TWh']:,.0f} TWh)"
    ),
    labels={"TWh": "Low-carbon energy (TWh)", "Country": "Country"}
)
fig2_bar.update_traces(
    texttemplate="%{x:,.0f} TWh",
    textposition="outside",
    cliponaxis=False
)
fig2_bar.update_layout(
    yaxis={"categoryorder": "total ascending"},
    margin=dict(t=80, l=10, r=70, b=10)
)
fig2_bar.show()

print("Leading country:")
print(leading_country[["Country", "TWh"]].to_string(index=False))

Leading country:
 France
 7593.0


### Task 2 — reflection

The **horizontal bar chart** communicates the ranking more clearly because all countries share a common baseline, so their TWh values and relative positions are easy to compare. The treemap is useful for showing each country's share of the overall low-carbon total, but comparing similarly sized rectangles is less precise.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [4]:
# Task 3 — regional fossil-energy mix (100% stacked horizontal bars)
# Reuse df_fossil created in Task 1.
regional_source = (
    df_fossil.groupby(["Region", "Source"], as_index=False)["TWh"]
    .sum()
)

regional_source["Region_Total"] = (
    regional_source.groupby("Region")["TWh"].transform("sum")
)
regional_source["Share"] = (
    regional_source["TWh"] / regional_source["Region_Total"] * 100
)

# Coal share is used to sort regions, highest first.
coal_share = (
    regional_source.loc[regional_source["Source"] == "Coal",
                        ["Region", "Share"]]
    .rename(columns={"Share": "Coal_Share"})
    .sort_values("Coal_Share", ascending=False)
)
region_order = coal_share["Region"].tolist()
most_coal_dependent = coal_share.iloc[0]

# Ensure all three source categories are consistently ordered.
source_order = ["Coal", "Oil", "Natural Gas"]
regional_source["Source"] = pd.Categorical(
    regional_source["Source"], categories=source_order, ordered=True
)
regional_source = regional_source.sort_values(["Region", "Source"])

# Only Coal receives a percentage label; Oil and Natural Gas remain unlabelled.
regional_source["Coal_Label"] = np.where(
    regional_source["Source"] == "Coal",
    regional_source["Share"].map(lambda x: f"{x:.1f}%"),
    ""
)

fig3 = px.bar(
    regional_source,
    x="Share",
    y="Region",
    color="Source",
    orientation="h",
    text="Coal_Label",
    category_orders={
        "Region": region_order,
        "Source": source_order
    },
    color_discrete_map={
        "Coal": fossil_palette["Coal"],
        "Oil": "#A6A6A6",
        "Natural Gas": "#D9D9D9"
    },
    title=(
        f"Coal is the largest fossil component in {most_coal_dependent['Region']} "
        f"({most_coal_dependent['Coal_Share']:.1f}% of fossil energy)"
    ),
    labels={"Share": "Share of regional fossil energy (%)", "Region": "Region"}
)
fig3.update_traces(
    textposition="inside",
    insidetextanchor="middle",
    hovertemplate=(
        "<b>%{y}</b><br>%{fullData.name}: %{x:.1f}%<extra></extra>"
    )
)
fig3.update_layout(
    barmode="stack",
    xaxis=dict(range=[0, 100], ticksuffix="%"),
    yaxis={"categoryorder": "array", "categoryarray": region_order[::-1]},
    legend_title_text="Fossil source",
    margin=dict(t=90, l=10, r=10, b=10)
)
fig3.show()

print("Most coal-dependent region:")
print(most_coal_dependent.to_string())

Most coal-dependent region:
Region           Africa
Coal_Share    78.039443


### Task 3 — reflection

The **treemap** answers: “Which countries and regions contribute the largest absolute amounts of each fossil fuel, and how is that total broken down?” The **100% stacked bar chart** answers: “What proportion of each region’s fossil-energy mix comes from coal, oil and natural gas, and which region has the highest coal share?” The treemap emphasizes absolute TWh; the stacked bar emphasizes composition.